# Berkshire Hathaway Sentiment Analysis

This notebook automatically loads sentiment features, downloads SPY market data, computes forward returns, and evaluates whether sentiment has predictive signal for the following year.


In [ ]:
from pathlib import Path
import sys

def get_project_root():
    cwd = Path.cwd().resolve()
    if cwd.name == "notebooks":
        return cwd.parent
    return cwd

BASE_DIR = get_project_root()
sys.path.insert(0, str(BASE_DIR / "src"))

import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

FEATURES_PATH = BASE_DIR / "features" / "sentiment_features.csv"

print(f"Loading data from: {FEATURES_PATH}")

df = pd.read_csv(FEATURES_PATH)
df = df.sort_values("year")

print("Loaded rows:", len(df))
print("Year range:", df["year"].min(), "-", df["year"].max())


In [ ]:
spy = yf.download("SPY", start="1995-01-01", progress=False)

if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)

spy["year"] = spy.index.year

yearly_returns = (
    spy.groupby("year")["Close"]
    .last()
    .pct_change()
    .rename("spy_return")
    .reset_index()
)

df = df.merge(yearly_returns, on="year", how="left")
df[["year", "sentiment_score", "spy_return"]].head()


In [ ]:
df["sentiment_ma_3"] = df["sentiment_score"].rolling(3).mean()

plt.figure()
plt.plot(df["year"], df["sentiment_score"], label="Raw")
plt.plot(df["year"], df["sentiment_ma_3"], label="3yr MA")
plt.legend()
plt.title("Sentiment (Raw vs Smoothed)")
plt.xlabel("Year")
plt.ylabel("Sentiment Score")
plt.show()


In [ ]:
plt.figure()
plt.scatter(df["sentiment_score"], df["spy_return"])
plt.title("Sentiment vs SPY Returns")
plt.xlabel("Sentiment")
plt.ylabel("SPY Return")
plt.show()


In [ ]:
df["next_year_return"] = df["spy_return"].shift(-1)

df_clean = df.dropna(subset=["sentiment_score", "spy_return", "next_year_return"]).copy()

print("\nSample data:")
print(df_clean[["year", "sentiment_score", "sentiment_ma_3", "next_year_return"]].head())


In [ ]:
df_clean["sentiment_bucket"] = pd.qcut(
    df_clean["sentiment_score"], 3, labels=["low", "mid", "high"]
)

results = df_clean.groupby("sentiment_bucket", observed=True)["next_year_return"].mean()

print("\n=== Forward Return by Sentiment Bucket ===")
print(results)

df_ma = df_clean.dropna(subset=["sentiment_ma_3"]).copy()
df_ma["sentiment_bucket_ma"] = pd.qcut(
    df_ma["sentiment_ma_3"], 3, labels=["low", "mid", "high"]
)

print("\n=== Forward Return (Smoothed Sentiment) ===")
print(
    df_ma.groupby("sentiment_bucket_ma", observed=True)["next_year_return"].mean()
)

print("\n=== Top/Bottom 20% Sentiment ===")
df_clean["extreme_bucket"] = pd.qcut(
    df_clean["sentiment_score"],
    5,
    labels=["very_low", "low", "mid", "high", "very_high"],
)

print(
    df_clean.groupby("extreme_bucket", observed=True)["next_year_return"].mean()
)


In [ ]:
print("\nCorrelation (sentiment vs next return):")
print(df_clean["sentiment_score"].corr(df_clean["next_year_return"]))

print("\nCorrelation (smoothed sentiment vs next return):")
print(df_ma["sentiment_ma_3"].corr(df_ma["next_year_return"]))


In [ ]:
print("\nINTERPRETATION GUIDE:")
print("- If LOW bucket > HIGH → contrarian signal")
print("- If HIGH bucket > LOW → momentum signal")
print("- If similar → weak/no signal")
